# Data Loading and Preparation

## Basic Imports

In [ ]:
# Data handling
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Pathing
from pathlib import Path

# Modelling
import joblib

# Metrics
from sklearn.metrics import classification_report

# Make plots look nicer
sns.set_theme(style="whitegrid")
%matplotlib inline

## Load Processed Feature Dataset for Evaluation

In [ ]:
split_path = Path("../data/splits")
X_test = pd.read_csv(split_path / "X_test.csv")
y_test = pd.read_csv(split_path / "y_test.csv")
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

### Cast to Categorical for Native Handling

In [ ]:
categorical_cols = ["state", "service", "dsport", "proto"]
for col in categorical_cols:
    X_test[col] = X_test[col].astype("category")

## Load Model From Disk

In [ ]:
model_path = Path("../models/xgboost_classifier_model.pkl")
model = joblib.load(model_path)
print(f"Model retrieved from {model_path}")

# Model Evaluation

## XGBoost Classifier Model

### Generate Predictions and Probabilities

In [ ]:
attack_prob = model.predict_proba(X_test)[:, 1]
pred_labels = model.predict(X_test)

### Evaluate and Visualise Predictions

#### Quick Stats

In [ ]:
num_attacks = pred_labels.sum()
print(f"Number of predicted attacks: {num_attacks}")

#### Classification Report

In [ ]:
print(classification_report(y_test, pred_labels))

#### Probability Distributions

In [ ]:
y_test_np = y_test.to_numpy().ravel()
tp_probs = attack_prob[y_test_np == 1]
fp_probs = attack_prob[y_test_np == 0]
plt.figure(figsize=(8, 6))
sns.kdeplot(tp_probs, label="True Positives (Label = 1)")
sns.kdeplot(fp_probs, label="False Positives (Label = 0)")
plt.legend()
plt.show()

#### Experiment with Threshold Tuning

In [ ]:
threshold = 0.67
custom_preds = (attack_prob >= threshold).astype(int)
print(classification_report(y_test, custom_preds))

Varying the decision threshold between 0.5 and 0.8 did not materially improve f1-scores or class balances (only shifting a small number of borderline cases). Suggests strong probability separation and limited gains.

### Inspect Model Feature Importances

#### Extract Importances

In [ ]:
features = [
    "dur",
    "Sintpkt",
    "sttl",
    "ct_state_ttl",
    "dbytes",
    "ttl_diff",
    "ttl_diff_bin",
    "sbytes",
    "Djit",
    "anomaly_score",
    "Dload",
    "Lhourofday",
    "state",
    "service",
    "dsport",
    "proto",
    "sttl_x_ct_state_ttl",
    "Shourofday"
]
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=False)
print(feat_imp)

First-pass model was heavily dominated by `sttl` (66% importance). Remaining features provided minor contributions. Strong reliance on a single feature may indicate potential sensitivity to feature drift.

Removing `sttl` during ablation testing had minimal impact on model performance, suggesting that the remaining features alone captured strong signal and effectively separated true positives from false positives. Additionally, feature importances became more balanced, with no single feature dominating.

#### Plot Importances

In [ ]:
feat_imp.sort_values(ascending=True).plot(kind="barh", figsize=(8, 6))
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("XGBoost Feature Importances")
plt.show()

### Mistake Analysis

#### Group Predictions

In [ ]:
# Ensure X_test aligned Series
y_test = pd.Series(y_test.squeeze(), index=X_test.index)
pred_labels = pd.Series(pred_labels, index=X_test.index)

# Define groups
fp_correct = X_test[(y_test == 0) & (pred_labels == 0)]
fp_misclassified = X_test[(y_test == 0) & (pred_labels == 1)]
tp_correct = X_test[(y_test == 1) & (pred_labels == 1)]

#### Compare Distributions

In [ ]:
df_plot = pd.concat([
    fp_correct.assign(Group="FP_Correct"),
    fp_misclassified.assign(Group="FP_Misclassified"),
    tp_correct.assign(Group="TP_Correct")
])
feature = "ttl_diff"
plt.figure(figsize=(8, 6))
sns.boxplot(data=df_plot, x="Group", y=feature)
plt.title(f"Distribution of {feature} by Group")